# 02 - Data Quality Assessment

**Purpose:** evaluate raw/interim data quality before cleaning. This notebook identifies missingness, placeholder values, invalid values, leakage risks, proxy-sensitive variables, and feature-governance issues.

This notebook owns **pre-clean quality assessment only**. It does not clean the dataset, encode features, perform modelling, or create the final feature matrix.

## Stage ownership

| Stage | This notebook owns | Moved elsewhere |
|---|---|---|
| Target distribution check | Yes | Model metrics later |
| Missingness diagnostics | Yes | Imputation in Notebook 03 / modelling pipeline |
| Missingness vs target | Yes | Model treatment later |
| Logical checks | Yes | Cleaning actions in Notebook 03 |
| Leakage/proxy review | Yes | Feature policy execution in Notebook 03/05 |
| Numeric/categorical profiling | Yes, diagnostic only | EDA storytelling in Notebook 04 |
| Statistical tests | No | Notebook 04 |
| Cleaning transformations | No | Notebook 03 |

In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
REPORT_DIR = PROJECT_ROOT / "reports"
TABLE_DIR = REPORT_DIR / "tables"
FIGURE_DIR = REPORT_DIR / "figures"

for path in [INTERIM_DIR, PROCESSED_DIR, TABLE_DIR, FIGURE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

TARGET_COL = "defaulter"

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


INTERIM_PATH = INTERIM_DIR / "credit_risk_merged_interim.csv"

if not INTERIM_PATH.exists():
    raise FileNotFoundError(
        f"Interim dataset not found at {INTERIM_PATH}. Run Notebook 01 or scripts/run_data_pipeline.py first."
    )

df = pd.read_csv(INTERIM_PATH, low_memory=False)
df.shape

(134417, 25)

## 1. Dataset overview

In [2]:
pre_clean_overview = pd.DataFrame({
    "metric": [
        "row_count",
        "column_count",
        "duplicate_full_rows",
        "target_present",
        "target_missing_count",
        "record_sequence_present",
        "record_key_duplicate_count",
    ],
    "value": [
        df.shape[0],
        df.shape[1],
        int(df.duplicated().sum()),
        TARGET_COL in df.columns,
        int(df[TARGET_COL].isna().sum()) if TARGET_COL in df.columns else np.nan,
        "record_sequence" in df.columns,
        int(df[["user_id", "record_sequence"]].duplicated().sum())
        if {"user_id", "record_sequence"}.issubset(df.columns)
        else np.nan,
    ],
})
pre_clean_overview.to_csv(TABLE_DIR / "02_pre_clean_overview.csv", index=False)
pre_clean_overview

,metric,value
0,row_count,134417
1,column_count,25
2,duplicate_full_rows,0
3,target_present,True
4,target_missing_count,0
5,record_sequence_present,True
6,record_key_duplicate_count,0


## 2. Target distribution and imbalance

In [3]:
if TARGET_COL not in df.columns:
    raise KeyError(f"Target column '{TARGET_COL}' was not found.")

target_distribution = (
    df[TARGET_COL]
    .value_counts(dropna=False)
    .rename_axis(TARGET_COL)
    .reset_index(name="row_count")
    .sort_values(TARGET_COL, na_position="last")
)
target_distribution["share_pct"] = (target_distribution["row_count"] / len(df) * 100).round(4)

target_quality_check = pd.DataFrame({
    "check": [
        "row_count",
        "target_missing_count",
        "default_count",
        "non_default_count",
        "default_rate_pct",
        "non_missing_unique_values",
        "binary_target_check",
    ],
    "result": [
        len(df),
        int(df[TARGET_COL].isna().sum()),
        int((df[TARGET_COL] == 1).sum()),
        int((df[TARGET_COL] == 0).sum()),
        round((df[TARGET_COL] == 1).mean() * 100, 4),
        sorted(df[TARGET_COL].dropna().unique().tolist()),
        set(df[TARGET_COL].dropna().unique()).issubset({0, 1}),
    ],
})

target_distribution.to_csv(TABLE_DIR / "02_target_distribution.csv", index=False)
target_quality_check.to_csv(TABLE_DIR / "02_target_quality_check.csv", index=False)

display(target_distribution)
display(target_quality_check)

,defaulter,row_count,share_pct
0,0,122264,90.9587
1,1,12153,9.0413


,check,result
0,row_count,134417
1,target_missing_count,0
2,default_count,12153
3,non_default_count,122264
4,default_rate_pct,9.0413
5,non_missing_unique_values,"[0, 1]"
6,binary_target_check,True


## 3. Missingness diagnostics

In [4]:
missingness_summary = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "row_count": len(df),
    "non_null_count": df.notna().sum().values,
    "missing_count": df.isna().sum().values,
    "missing_pct": (df.isna().mean().values * 100).round(2),
    "unique_values": df.nunique(dropna=True).values,
})
missingness_summary["complete_pct"] = (100 - missingness_summary["missing_pct"]).round(2)
missingness_summary["missingness_severity"] = pd.cut(
    missingness_summary["missing_pct"],
    bins=[-0.01, 0, 5, 20, 50, 100],
    labels=["No missingness", "Low missingness", "Moderate missingness", "High missingness", "Severe missingness"],
)

missingness_with_gaps = missingness_summary.query("missing_count > 0").copy()
missingness_with_gaps = missingness_with_gaps.sort_values(
    ["missing_pct", "missing_count"], ascending=False
).reset_index(drop=True)

missingness_summary.to_csv(TABLE_DIR / "02_missingness_summary_all_columns.csv", index=False)
missingness_with_gaps.to_csv(TABLE_DIR / "02_missingness_summary_missing_only.csv", index=False)

missingness_with_gaps

,column,dtype,row_count,non_null_count,missing_count,missing_pct,unique_values,complete_pct,missingness_severity
0,employment_type,object,134417,54731,79686,59.2800,2,40.7200,Severe missingness
1,tier_of_employment,object,134417,54731,79686,59.2800,7,40.7200,Severe missingness
2,married,object,134417,89333,45084,33.5400,2,66.4600,High missingness
3,social_profile,object,134417,89571,44846,33.3600,2,66.6400,High missingness
4,is_verified,object,134417,100910,33507,24.9300,3,75.0700,High missingness
5,amount,float64,134417,106798,27619,20.5500,86157,79.4500,High missingness
6,industry,object,134417,134413,4,0.0000,12974,100.0000,No missingness
7,work_experience,object,134417,134413,4,0.0000,7,100.0000,No missingness


## 4. Missingness by target

In [5]:
feature_cols = [col for col in df.columns if col != TARGET_COL]
class_counts = df[TARGET_COL].value_counts(dropna=False).reindex([0, 1], fill_value=0)

missing_count_by_target = (
    df[feature_cols]
    .isna()
    .groupby(df[TARGET_COL])
    .sum()
    .T
    .reindex(columns=[0, 1], fill_value=0)
    .rename(columns={0: "missing_count_non_default", 1: "missing_count_default"})
    .reset_index()
    .rename(columns={"index": "column"})
)
missing_count_by_target.columns.name = None

missing_count_by_target["non_default_row_count"] = int(class_counts.loc[0])
missing_count_by_target["default_row_count"] = int(class_counts.loc[1])
missing_count_by_target["missing_rate_non_default"] = (
    missing_count_by_target["missing_count_non_default"] / missing_count_by_target["non_default_row_count"] * 100
).round(2)
missing_count_by_target["missing_rate_default"] = (
    missing_count_by_target["missing_count_default"] / missing_count_by_target["default_row_count"] * 100
).round(2)
missing_count_by_target["absolute_gap"] = (
    missing_count_by_target["missing_rate_default"] - missing_count_by_target["missing_rate_non_default"]
).abs().round(2)
missing_count_by_target["total_missing_count"] = (
    missing_count_by_target["missing_count_non_default"] + missing_count_by_target["missing_count_default"]
)

missingness_by_target = (
    missing_count_by_target
    .query("total_missing_count > 0")
    .sort_values(["absolute_gap", "total_missing_count"], ascending=False)
    .reset_index(drop=True)
)

missingness_by_target.to_csv(TABLE_DIR / "02_missingness_by_target.csv", index=False)
missingness_by_target

,column,missing_count_non_default,missing_count_default,non_default_row_count,default_row_count,missing_rate_non_default,missing_rate_default,absolute_gap,total_missing_count
0,amount,23165,4454,122264,12153,18.9500,36.6500,17.7000,27619
1,employment_type,73003,6683,122264,12153,59.7100,54.9900,4.7200,79686
2,tier_of_employment,73003,6683,122264,12153,59.7100,54.9900,4.7200,79686
3,social_profile,40849,3997,122264,12153,33.4100,32.8900,0.5200,44846
4,married,40962,4122,122264,12153,33.5000,33.9200,0.4200,45084
5,is_verified,30450,3057,122264,12153,24.9100,25.1500,0.2400,33507
6,industry,4,0,122264,12153,0.0000,0.0000,0.0000,4
7,work_experience,4,0,122264,12153,0.0000,0.0000,0.0000,4


## 5. Numeric field profiling

In [6]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()
numeric_review = []

for col in numeric_cols:
    s = df[col]
    non_null = s.dropna()
    n = len(non_null)
    unique_count = non_null.nunique()
    unique_ratio = unique_count / n if n > 0 else 0
    is_integer_like = pd.api.types.is_integer_dtype(s) or (
        n > 0 and np.all(np.isclose(non_null, np.round(non_null)))
    )
    unique_values = set(non_null.unique()) if n > 0 else set()
    is_binary = unique_values.issubset({0, 1})
    is_likely_id = unique_ratio >= 0.95 and is_integer_like
    is_low_cardinality_integer = unique_count <= 15 and is_integer_like
    include_in_math_profile = not (is_binary or is_likely_id or is_low_cardinality_integer)

    if is_binary:
        inferred_type = "binary_flag"
    elif is_likely_id:
        inferred_type = "likely_identifier"
    elif is_low_cardinality_integer:
        inferred_type = "low_cardinality_numeric_or_ordinal"
    else:
        inferred_type = "mathematical_numeric"

    numeric_review.append({
        "column": col,
        "dtype": str(s.dtype),
        "non_null_count": int(s.notna().sum()),
        "missing_count": int(s.isna().sum()),
        "missing_pct": round(s.isna().mean() * 100, 2),
        "unique_values": int(unique_count),
        "unique_ratio": round(unique_ratio, 4),
        "min": non_null.min() if n > 0 else np.nan,
        "max": non_null.max() if n > 0 else np.nan,
        "inferred_type": inferred_type,
        "include_in_math_profile": include_in_math_profile,
    })

numeric_column_review = pd.DataFrame(numeric_review)
math_numeric_cols = numeric_column_review.loc[
    numeric_column_review["include_in_math_profile"], "column"
].tolist()

numeric_profile = pd.DataFrame()
if math_numeric_cols:
    numeric_profile = (
        df[math_numeric_cols]
        .agg(["count", "mean", "median", "std", "min", "max", "skew"])
        .T
        .reset_index()
        .rename(columns={"index": "column"})
    )
    numeric_profile["missing_count"] = df[math_numeric_cols].isna().sum().values
    numeric_profile["missing_pct"] = (df[math_numeric_cols].isna().mean().values * 100).round(2)
    numeric_profile = numeric_profile.round(2)

numeric_column_review.to_csv(TABLE_DIR / "02_numeric_column_review.csv", index=False)
numeric_profile.to_csv(TABLE_DIR / "02_numeric_profile.csv", index=False, float_format="%.2f")

display(numeric_column_review)
display(numeric_profile)

,column,dtype,non_null_count,missing_count,missing_pct,unique_values,unique_ratio,min,max,inferred_type,include_in_math_profile
0,user_id,int64,134417,0,0.0000,133752,0.9951,"208,036.0000","78,958,941.0000",likely_identifier,False
1,amount,float64,106798,27619,20.5500,86157,0.8067,0.0000,"8,000,078.0000",mathematical_numeric,True
2,interest_rate,float64,134417,0,0.0000,137,0.0010,5.4200,23.5400,mathematical_numeric,True
3,tenure_years,int64,134417,0,0.0000,2,0.0000,4.0000,6.0000,low_cardinality_numeric_or_ordinal,False
4,record_sequence,int64,134417,0,0.0000,2,0.0000,1.0000,2.0000,low_cardinality_numeric_or_ordinal,False
5,total_income_pa,float64,134417,0,0.0000,11380,0.0847,"4,000.0000","7,141,778.0000",mathematical_numeric,True
6,dependents,int64,134417,0,0.0000,5,0.0000,0.0000,4.0000,low_cardinality_numeric_or_ordinal,False
7,delinq_2yrs,int64,134417,0,0.0000,22,0.0002,0.0000,22.0000,mathematical_numeric,True
8,total_payment,float64,134417,0,0.0000,104202,0.7752,0.0000,"57,777.5799",mathematical_numeric,True
9,received_principal,float64,134417,0,0.0000,42022,0.3126,0.0000,"35,000.0100",mathematical_numeric,True


,column,count,mean,median,std,min,max,skew,missing_count,missing_pct
0,amount,"106,798.0000","137,629.1700","76,503.0000","157,596.6500",0.0000,"8,000,078.0000",4.3500,27619,20.5500
1,interest_rate,"134,417.0000",12.0200,11.8400,3.8800,5.4200,23.5400,0.3600,0,0.0000
2,total_income_pa,"134,417.0000","72,597.9800","62,000.0000","56,100.0500","4,000.0000","7,141,778.0000",28.7400,0,0.0000
3,delinq_2yrs,"134,417.0000",0.2800,0.0000,0.8000,0.0000,22.0000,5.4200,0,0.0000
4,total_payment,"134,417.0000","10,696.4400","8,061.1400","8,544.3100",0.0000,"57,777.5800",1.6000,0,0.0000
5,received_principal,"134,417.0000","8,282.6800","5,869.1200","7,184.0200",0.0000,"35,000.0100",1.5600,0,0.0000
6,interest_received,"134,417.0000","2,335.8700","1,627.0300","2,419.9100",0.0000,"24,205.6200",2.8500,0,0.0000


## 6. Logical data-quality checks

In [7]:
logical_checks = []

def add_check(check_name, condition, columns_checked=None, issue_type=None, recommended_action=None):
    condition = condition.fillna(False) if hasattr(condition, "fillna") else condition
    failed_count = int(condition.sum())
    failed_pct = round(failed_count / len(df) * 100, 4) if len(df) else 0
    logical_checks.append({
        "check": check_name,
        "columns_checked": ", ".join(columns_checked) if columns_checked else None,
        "issue_type": issue_type,
        "failed_row_count": failed_count,
        "failed_row_pct": failed_pct,
        "status": "Fail" if failed_count > 0 else "Pass",
        "recommended_action": recommended_action,
    })

if "amount" in df.columns:
    add_check("amount_missing", df["amount"].isna(), ["amount"], "missing_value", "Create flag and retain rows.")
    add_check("amount_non_positive", df["amount"].notna() & (df["amount"] <= 0), ["amount"], "invalid_value", "Flag and convert to missing in cleaning.")

if "interest_rate" in df.columns:
    add_check("interest_rate_outside_0_to_100", df["interest_rate"].notna() & ((df["interest_rate"] <= 0) | (df["interest_rate"] > 100)), ["interest_rate"], "invalid_range", "Review invalid interest-rate values.")

if "tenure_years" in df.columns:
    add_check("tenure_years_non_positive", df["tenure_years"].notna() & (df["tenure_years"] <= 0), ["tenure_years"], "invalid_value", "Review invalid tenure values.")

if "total_income_pa" in df.columns:
    add_check("income_non_positive", df["total_income_pa"].notna() & (df["total_income_pa"] <= 0), ["total_income_pa"], "invalid_value", "Review income values.")

for col in ["dependents", "total_payment", "received_principal", "interest_received", "number_of_loans"]:
    if col in df.columns:
        add_check(f"{col}_negative", df[col].notna() & (df[col] < 0), [col], "invalid_value", "Negative values require review.")

if {"received_principal", "amount"}.issubset(df.columns):
    add_check(
        "received_principal_greater_than_amount_when_amount_present",
        df["received_principal"].notna() & df["amount"].notna() & (df["received_principal"] > df["amount"]),
        ["received_principal", "amount"],
        "business_rule_exception",
        "Document and avoid repayment-derived fields in baseline modelling.",
    )

if {"total_payment", "received_principal"}.issubset(df.columns):
    add_check(
        "total_payment_less_than_received_principal",
        df["total_payment"].notna() & df["received_principal"].notna() & (df["total_payment"] < df["received_principal"]),
        ["total_payment", "received_principal"],
        "business_rule_exception",
        "Review repayment construction and timing.",
    )

logical_quality_checks = (
    pd.DataFrame(logical_checks)
    .sort_values(["failed_row_count", "failed_row_pct"], ascending=False)
    .reset_index(drop=True)
)
failed_logical_quality_checks = logical_quality_checks.query("failed_row_count > 0").reset_index(drop=True)

logical_quality_checks.to_csv(TABLE_DIR / "02_logical_quality_checks_all.csv", index=False)
failed_logical_quality_checks.to_csv(TABLE_DIR / "02_logical_quality_checks_failed.csv", index=False)

failed_logical_quality_checks

,check,columns_checked,issue_type,failed_row_count,failed_row_pct,status,recommended_action
0,amount_missing,amount,missing_value,27619,20.5473,Fail,Create flag and retain rows.
1,received_principal_greater_than_amount_when_amount_present,"received_principal, amount",business_rule_exception,2815,2.0942,Fail,Document and avoid repayment-derived fields in baseline modelling.
2,amount_non_positive,amount,invalid_value,19,0.0141,Fail,Flag and convert to missing in cleaning.


## 7. Categorical profile and value distribution

In [8]:
categorical_cols = df.select_dtypes(include=["object", "category", "string", "bool"]).columns.tolist()

categorical_profile_rows = []
categorical_value_tables = []

for col in categorical_cols:
    s = df[col]
    value_counts = s.value_counts(dropna=True, normalize=True)
    categorical_profile_rows.append({
        "column": col,
        "dtype": str(s.dtype),
        "row_count": len(df),
        "non_null_count": int(s.notna().sum()),
        "missing_count": int(s.isna().sum()),
        "missing_pct": round(s.isna().mean() * 100, 2),
        "unique_values": int(s.nunique(dropna=True)),
        "top_value": value_counts.index[0] if not value_counts.empty else np.nan,
        "top_value_share_pct": round(value_counts.iloc[0] * 100, 2) if not value_counts.empty else np.nan,
    })

    temp = df[[col, TARGET_COL]].copy() if TARGET_COL in df.columns else df[[col]].copy()
    temp[col] = temp[col].astype("object").where(temp[col].notna(), "Missing")
    grouped = temp.groupby(col, dropna=False).size().reset_index(name="row_count").rename(columns={col: "category_value"})
    grouped["column"] = col
    grouped["share_pct"] = (grouped["row_count"] / len(df) * 100).round(2)
    if TARGET_COL in df.columns:
        defaults = temp.groupby(col, dropna=False)[TARGET_COL].sum().reset_index(name="default_count").rename(columns={col: "category_value"})
        grouped = grouped.merge(defaults, on="category_value", how="left")
        grouped["default_rate_pct"] = (grouped["default_count"] / grouped["row_count"] * 100).round(2)
    else:
        grouped["default_count"] = np.nan
        grouped["default_rate_pct"] = np.nan
    grouped["is_missing_category"] = grouped["category_value"].eq("Missing")
    grouped["rare_category_flag"] = grouped["share_pct"] < 1
    categorical_value_tables.append(grouped[[
        "column", "category_value", "row_count", "share_pct",
        "default_count", "default_rate_pct", "is_missing_category", "rare_category_flag"
    ]])

categorical_profile = pd.DataFrame(categorical_profile_rows)
if not categorical_profile.empty:
    categorical_profile["cardinality_level"] = pd.cut(
        categorical_profile["unique_values"],
        bins=[-1, 1, 10, 50, 500, np.inf],
        labels=["single_value", "low_cardinality", "medium_cardinality", "high_cardinality", "very_high_cardinality"],
    )
    categorical_profile = categorical_profile.sort_values(["unique_values", "missing_pct"], ascending=False)

categorical_value_distribution = pd.concat(categorical_value_tables, ignore_index=True) if categorical_value_tables else pd.DataFrame()
if not categorical_value_distribution.empty:
    categorical_value_distribution = categorical_value_distribution.sort_values(["column", "row_count"], ascending=[True, False])

categorical_profile.to_csv(TABLE_DIR / "02_categorical_profile.csv", index=False)
categorical_value_distribution.to_csv(TABLE_DIR / "02_categorical_value_distribution.csv", index=False)

display(categorical_profile)
display(categorical_value_distribution.head(40))

,column,dtype,row_count,non_null_count,missing_count,missing_pct,unique_values,top_value,top_value_share_pct,cardinality_level
3,industry,object,134417,134413,4,0.0000,12974,0,84.5000,very_high_cardinality
9,pincode,object,134417,134417,0,0.0000,844,XX112X,1.1800,very_high_cardinality
4,role,object,134417,134417,0,0.0000,46,KHMbckjadbckIFGCASEWdkcndwkcnCCM,15.5300,medium_cardinality
2,tier_of_employment,object,134417,54731,79686,59.2800,7,B,30.8600,low_cardinality
0,loan_category,object,134417,134417,0,0.0000,7,Consolidation,60.1600,low_cardinality
5,work_experience,object,134417,134413,4,0.0000,7,0,84.5000,low_cardinality
8,home,object,134417,134417,0,0.0000,5,mortgage,48.7200,low_cardinality
11,is_verified,object,134417,100910,33507,24.9300,3,Not Verified,33.4200,low_cardinality
6,gender,object,134417,134417,0,0.0000,3,Female,33.4000,low_cardinality
1,employment_type,object,134417,54731,79686,59.2800,2,Salaried,81.8900,low_cardinality


,column,category_value,row_count,share_pct,default_count,default_rate_pct,is_missing_category,rare_category_flag
7,employment_type,Missing,79686,59.2800,6683,8.3900,True,False
8,employment_type,Salaried,44817,33.3400,4019,8.9700,False,False
9,employment_type,Self - Employeed,9914,7.3800,1451,14.6400,False,False
13047,gender,Female,44898,33.4000,4059,9.0400,False,False
13049,gender,Other,44850,33.3700,4091,9.1200,False,False
13048,gender,Male,44669,33.2300,4003,8.9600,False,False
13053,home,mortgage,65490,48.7200,5204,7.9500,False,False
13057,home,rent,56442,41.9900,5853,10.3700,False,False
13056,home,own,12397,9.2200,1080,8.7100,False,False
13055,home,other,46,0.0300,9,19.5700,False,True


## 8. Leakage, timing, and sensitive/proxy review

In [9]:
leakage_review = pd.DataFrame([
    {"feature": "defaulter", "risk_level": "target", "risk_type": "target", "baseline_model_policy": "exclude_target", "reason": "Target variable; never used as predictor."},
    {"feature": "total_payment", "risk_level": "high", "risk_type": "post_origination_repayment", "baseline_model_policy": "exclude_from_baseline", "reason": "Repayment amount may include post-origination performance and leak target outcome."},
    {"feature": "received_principal", "risk_level": "high", "risk_type": "post_origination_repayment", "baseline_model_policy": "exclude_from_baseline", "reason": "Principal received may be observed after loan performance begins."},
    {"feature": "interest_received", "risk_level": "high", "risk_type": "post_origination_repayment", "baseline_model_policy": "exclude_from_baseline", "reason": "Interest received may be post-outcome repayment behaviour."},
    {"feature": "delinq_2yrs", "risk_level": "medium", "risk_type": "timing_dependent_credit_history", "baseline_model_policy": "review_timing_before_use", "reason": "Valid only if available at prediction point."},
    {"feature": "number_of_loans", "risk_level": "medium", "risk_type": "timing_dependent_exposure", "baseline_model_policy": "review_timing_before_use", "reason": "Could be valid exposure information, but timing must be confirmed."},
])

fairness_proxy_review = pd.DataFrame([
    {"feature": "gender", "risk_level": "high", "risk_type": "sensitive_attribute", "baseline_model_policy": "exclude_from_baseline", "reason": "Sensitive/protected characteristic; retain only for governance/fairness diagnostics if appropriate."},
    {"feature": "pincode", "risk_level": "high", "risk_type": "geographic_proxy", "baseline_model_policy": "exclude_from_baseline", "reason": "Geographic proxy that may encode socioeconomic patterns."},
    {"feature": "social_profile", "risk_level": "high", "risk_type": "unclear_social_proxy", "baseline_model_policy": "exclude_from_baseline", "reason": "Unclear meaning and potential social/behavioural proxy."},
    {"feature": "married", "risk_level": "medium", "risk_type": "household_status_proxy", "baseline_model_policy": "governance_review_before_use", "reason": "Household/social status proxy."},
    {"feature": "dependents", "risk_level": "medium", "risk_type": "family_status_proxy", "baseline_model_policy": "governance_review_before_use", "reason": "Family status proxy; use cautiously."},
    {"feature": "home", "risk_level": "medium", "risk_type": "socioeconomic_proxy", "baseline_model_policy": "governance_review_before_use", "reason": "Wealth/socioeconomic proxy; common in credit risk but requires monitoring."},
    {"feature": "is_verified", "risk_level": "medium", "risk_type": "operational_process_proxy", "baseline_model_policy": "governance_review_before_use", "reason": "Potential verification/operational bias."},
])

feature_governance_review = pd.concat(
    [
        leakage_review.assign(review_area="Leakage / timing review"),
        fairness_proxy_review.assign(review_area="Sensitive / proxy review"),
    ],
    ignore_index=True,
)

leakage_review.to_csv(TABLE_DIR / "02_leakage_review.csv", index=False)
fairness_proxy_review.to_csv(TABLE_DIR / "02_fairness_proxy_review.csv", index=False)
feature_governance_review.to_csv(TABLE_DIR / "02_feature_governance_review.csv", index=False)

feature_governance_review

,feature,risk_level,risk_type,baseline_model_policy,reason,review_area
0,defaulter,target,target,exclude_target,Target variable; never used as predictor.,Leakage / timing review
1,total_payment,high,post_origination_repayment,exclude_from_baseline,Repayment amount may include post-origination performance and leak target outcome.,Leakage / timing review
2,received_principal,high,post_origination_repayment,exclude_from_baseline,Principal received may be observed after loan performance begins.,Leakage / timing review
3,interest_received,high,post_origination_repayment,exclude_from_baseline,Interest received may be post-outcome repayment behaviour.,Leakage / timing review
4,delinq_2yrs,medium,timing_dependent_credit_history,review_timing_before_use,Valid only if available at prediction point.,Leakage / timing review
5,number_of_loans,medium,timing_dependent_exposure,review_timing_before_use,"Could be valid exposure information, but timing must be confirmed.",Leakage / timing review
6,gender,high,sensitive_attribute,exclude_from_baseline,Sensitive/protected characteristic; retain only for governance/fairness diagnostics if appropriate.,Sensitive / proxy review
7,pincode,high,geographic_proxy,exclude_from_baseline,Geographic proxy that may encode socioeconomic patterns.,Sensitive / proxy review
8,social_profile,high,unclear_social_proxy,exclude_from_baseline,Unclear meaning and potential social/behavioural proxy.,Sensitive / proxy review
9,married,medium,household_status_proxy,governance_review_before_use,Household/social status proxy.,Sensitive / proxy review


## 9. Data-quality decision log for Notebook 03

In [10]:
decision_log = pd.DataFrame([
    {"area": "Observation grain", "decision": "Preserve user_id + record_sequence.", "next_stage": "Notebook 03 cleaning"},
    {"area": "Missing amount", "decision": "Do not drop rows; create amount_missing_flag and impute later in modelling pipeline.", "next_stage": "Notebook 03 / 06"},
    {"area": "Invalid amount", "decision": "Flag non-positive amount and convert to missing in cleaning.", "next_stage": "Notebook 03"},
    {"area": "Employment missingness", "decision": "Create flags and fill categorical missing values as Unknown.", "next_stage": "Notebook 03"},
    {"area": "Placeholder zero", "decision": "Treat placeholder 0 in industry/work_experience as Unknown/unavailable.", "next_stage": "Notebook 03"},
    {"area": "Leakage fields", "decision": "Exclude repayment-derived fields from baseline model; retain for monitoring.", "next_stage": "Notebook 03 / 04 / 05"},
    {"area": "Sensitive/proxy variables", "decision": "Retain for governance; exclude high-risk fields from baseline model.", "next_stage": "Notebook 03 / 05"},
    {"area": "Preprocessing leakage", "decision": "Do not fit encoders, scalers, resamplers, or transformations before train/test split.", "next_stage": "Notebook 06"},
])

decision_log.to_csv(TABLE_DIR / "02_data_quality_decision_log_for_notebook_03.csv", index=False)
decision_log

,area,decision,next_stage
0,Observation grain,Preserve user_id + record_sequence.,Notebook 03 cleaning
1,Missing amount,Do not drop rows; create amount_missing_flag and impute later in modelling pipeline.,Notebook 03 / 06
2,Invalid amount,Flag non-positive amount and convert to missing in cleaning.,Notebook 03
3,Employment missingness,Create flags and fill categorical missing values as Unknown.,Notebook 03
4,Placeholder zero,Treat placeholder 0 in industry/work_experience as Unknown/unavailable.,Notebook 03
5,Leakage fields,Exclude repayment-derived fields from baseline model; retain for monitoring.,Notebook 03 / 04 / 05
6,Sensitive/proxy variables,Retain for governance; exclude high-risk fields from baseline model.,Notebook 03 / 05
7,Preprocessing leakage,"Do not fit encoders, scalers, resamplers, or transformations before train/test split.",Notebook 06


## Carry-forward decisions

Notebook 03 should apply only approved cleaning actions and create a clean, auditable dataset. Notebook 04 should use the cleaned dataset for EDA, statistical analysis, and portfolio monitoring.